# 27 — Inside one transformer block

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarahmedZa/ai-ml-course/blob/main/notebooks/027-transformer-block.ipynb)

A transformer block is two small networks wrapped in two shortcuts: attention mixes information between tokens, the feed-forward layer thinks about each token alone, and the residual adds keep the original signal alive. You will run all four pieces in NumPy, then measure what the shortcuts are worth.

## 1. The numbers

Two tokens, four numbers each. Every weight matrix is a table of numbers, and every layer here is a matrix multiply followed by a normalisation. The two output projections are deliberately tiny, so a freshly initialised block barely disturbs the stream.

In [ ]:
import numpy as np                                     # the array library, already on Colab

rng = np.random.default_rng(4)                        # fixed seed, so your numbers match mine
d = 4                                                 # width of a token vector
X = rng.normal(size=(2, d))                           # two tokens, four numbers each
Wq = rng.normal(scale=0.5, size=(d, d))               # query projection
Wk = rng.normal(scale=0.5, size=(d, d))               # key projection
Wv = rng.normal(scale=0.5, size=(d, d))               # value projection
Wo = rng.normal(scale=0.02, size=(d, d))              # attention output projection (small)
W1 = rng.normal(scale=0.5, size=(d, d))               # feed-forward up projection
W2 = rng.normal(scale=0.02, size=(d, d))              # feed-forward down projection (small)

def ln(h):                                            # layer norm: one token per row
    mu = h.mean(axis=-1, keepdims=True)               # the mean of that token's numbers
    sd = h.std(axis=-1, keepdims=True)                # their spread
    return (h - mu) / (sd + 1e-5)                     # unit scale, small guard against zero

def attn(h):                                          # multi-head attention, one head
    q = h @ Wq                                        # what each token is looking for
    k = h @ Wk                                        # what each token advertises
    v = h @ Wv                                        # what each token hands over
    s = q @ k.T / np.sqrt(d)                          # every query against every key
    p = np.exp(s - s.max(axis=-1, keepdims=True))     # exponentials, max subtracted for safety
    p = p / p.sum(axis=-1, keepdims=True)             # each row is a share of one
    return p @ v @ Wo                                 # mix the values, then project out

def ffn(h):                                           # the per-token thinking layer
    return np.maximum(h @ W1, 0) @ W2                 # expand, cut negatives, contract

A = attn(ln(X))                                       # attention sees normalised tokens
H1 = X + A                                            # residual: add, do not replace
F = ffn(ln(H1))                                       # feed-forward on the updated stream
H2 = H1 + F                                           # second residual

S = (ln(X) @ Wq) @ (ln(X) @ Wk).T / np.sqrt(d)        # the score table, recomputed for printing
P = np.exp(S - S.max(axis=-1, keepdims=True))         # exponentials of the scores
P = P / P.sum(axis=-1, keepdims=True)                 # each row becomes a share of one
print("attention weights:", np.round(P, 3))          # who looks at whom
print("|input|", round(np.linalg.norm(X), 2))         # about 2.6
print("|attention|", round(np.linalg.norm(A), 3))     # tiny: the small output projection
print("|after add|", round(np.linalg.norm(H1), 2))    # still about 2.6
print("|after ffn|", round(np.linalg.norm(H2), 2))    # again about 2.6

**Line by line**

- `rng = np.random.default_rng(0)` — a seeded random generator, so results repeat.
- `d = 4` — every token vector is four numbers wide.
- `X` — the input: two tokens as two rows.
- `Wq`, `Wk`, `Wv` — three projection tables that turn a token into query, key, value.
- `Wo`, `W2` — output projections with small entries, so the branch stays quiet.
- `ln(h)` — subtract the row mean, divide by the row spread, one row per token.
- `q = h @ Wq` — a matrix multiply applies the table to every token at once.
- `s = q @ k.T / np.sqrt(d)` — all query-key dot products, scaled down.
- `p = e / e.sum(...)` — softmax over each row, so each token's weights sum to one.
- `return p @ v @ Wo` — average the values, then project back to width `d`.
- `ffn(h)` — up-project, keep only positive entries, project back down.
- `H1 = X + A` — the residual: the block adds to the stream instead of replacing it.
- `F = ffn(ln(H1))` and `H2 = H1 + F` — the feed-forward branch, added the same way.
- `S`, `P` — the score table and its softmax, recomputed so we can print the weights.
- `print(...)` — the weights, then the length of the signal after each stage.

## 2. Step by step

Same block, one token at a time, printing the size of the signal after each stage. Watch how the two numbers that matter are the branch's norm (small) and the stream's norm (unchanged).

In [ ]:
N = ln(X)                                             # normalise both tokens once
K = N @ Wk                                            # all keys
V = N @ Wv                                            # all values

for i in range(2):                                    # one pass per token
    q_i = N[i] @ Wq                                   # this token's query
    s_i = K @ q_i / np.sqrt(d)                        # its scores against both keys
    e_i = np.exp(s_i - s_i.max())                     # stable exponentials
    w_i = e_i / e_i.sum()                             # its attention weights
    mix = w_i @ V @ Wo                                # the weighted sum, projected out
    h = X[i] + mix                                    # add the branch to the stream
    n = ln(h)                                         # normalise before the feed-forward
    out = h + ffn(n)                                  # second add
    print(f"token {i}: weights={np.round(w_i, 3)} branch={np.linalg.norm(mix):.3f} "   # first half
          f"stream={np.linalg.norm(out):.2f}")        # one line per token

print("all tokens moved by", np.round(np.linalg.norm(H2 - X), 3))   # the block's total effect

**Line by line**

- `N = ln(X)` — normalise both tokens once, outside the loop.
- `K`, `V` — every token's key and value, computed in one multiply.
- `for i in range(2)` — one iteration per token.
- `q_i = N[i] @ Wq` — the query of this token only.
- `s_i = K @ q_i / np.sqrt(d)` — its score against each key.
- `w_i = e_i / e_i.sum()` — softmax over those two scores.
- `mix = w_i @ V @ Wo` — blend the values and project back to width `d`.
- `h = X[i] + mix` — the residual add: the token keeps its own numbers.
- `out = h + ffn(n)` — the feed-forward branch is added the same way.
- `print(f"...")` — an f-string; `:.3f` keeps three decimals.

## 3. Plot: where the magnitude lives

Each stage of the block produces a vector, and we can measure its length. Bars for a block with the residual, next to bars for the same block with the shortcut removed.

In [ ]:
import matplotlib.pyplot as plt                       # plotting library, already on Colab

def stage_norms(residual):                            # lengths after each stage of one block
    n0 = ln(X)                                        # the normalised input
    a = attn(n0)                                      # the attention branch alone
    h = X + a if residual else a                      # add it, or let it replace the stream
    f = ffn(ln(h))                                    # the feed-forward branch
    h = h + f if residual else f                      # add it, or replace again
    return [np.linalg.norm(X), np.linalg.norm(a), np.linalg.norm(X + a if residual else a),   # three
            np.linalg.norm(ln(h)), np.linalg.norm(h)]  # five lengths, in stage order

stages = ["input", "attention", "after add", "after LN", "after FFN"]   # x-axis labels
with_res = stage_norms(True)                          # the block as designed
no_res = stage_norms(False)                           # the block with no shortcut
idx = np.arange(len(stages))                          # one slot per stage

plt.figure(figsize=(6, 3))                            # a wide, short figure
plt.bar(idx - 0.2, with_res, width=0.4, label="with residual")     # left bar of each pair
plt.bar(idx + 0.2, no_res, width=0.4, label="without residual")    # right bar of each pair
plt.xticks(idx, stages, rotation=15)                  # name the stages
plt.ylabel("total magnitude (L2 norm)")               # what the bars measure
plt.legend()                                          # name the two colours
plt.title("the shortcut is what keeps the signal big")   # the takeaway
plt.show()                                            # render it

**Line by line**

- `import matplotlib.pyplot as plt` — load the plotting library.
- `def stage_norms(residual)` — returns five lengths in stage order.
- `a = attn(n0)` — the attention branch's output on its own.
- `h = X + a if residual else a` — add to the stream, or overwrite it.
- `f = ffn(ln(h))` — run the feed-forward branch on the result.
- `return [...]` — the input, the branch, the sum, the normalised sum, the final state.
- `plt.bar(idx - 0.2, ...)` — bars nudged left, so the pair sits side by side.
- `plt.xticks(idx, stages, rotation=15)` — put the stage names under the pairs.

## 4. Interactive: slide the depth

Stack the block and watch the stream's length at every depth. Each point is averaged over sixteen random inputs, so one lucky direction cannot decide the picture.

In [ ]:
from ipywidgets import interact                       # sliders that rerun a function

def trace(depth, residual, n_inputs=16):              # average magnitude at every depth
    total = np.zeros(depth + 1)                       # one slot per depth, plus the input
    for s in range(n_inputs):                         # average over several starting points
        h = np.random.default_rng(100 + s).normal(size=(2, d))   # a fresh input
        total[0] += np.linalg.norm(h)                 # the input's length
        for step in range(depth):                     # apply the block `depth` times
            a = attn(ln(h))                           # attention branch
            h = h + a if residual else a              # add, or overwrite
            f = ffn(ln(h))                            # feed-forward branch
            h = h + f if residual else f              # add, or overwrite
            total[step + 1] += np.linalg.norm(h)      # record the length
    return total / n_inputs                           # the average curve

def show(depth=6):                                    # depth comes from the slider
    w = trace(depth, True)                            # with the shortcut
    o = trace(depth, False)                           # without it
    plt.figure(figsize=(5, 3))                        # a fresh figure on every drag
    plt.plot(w, marker="o", label="with residual")    # the healthy curve
    plt.plot(o, marker="o", label="without residual") # the collapsing curve
    plt.yscale("log")                                 # collapse spans many factors
    plt.xlabel("block")                               # x-axis label
    plt.ylabel("magnitude (log scale)")               # y-axis label
    plt.legend()                                      # name the two lines
    plt.title(f"depth {depth}: {w[-1]:.2f} with, {o[-1]:.3f} without")   # live readout
    plt.show()                                        # draw the frame

interact(show, depth=(1, 24, 1))                      # attach the slider

**Line by line**

- `from ipywidgets import interact` — Colab ships the widget library.
- `def trace(depth, residual, n_inputs=16)` — average the curve over many inputs.
- `total = np.zeros(depth + 1)` — an accumulator with one slot per depth.
- `h = np.random.default_rng(100 + s).normal(...)` — a different starting input each time.
- `for step in range(depth)` — apply the block the requested number of times.
- `h = h + a if residual else a` — the only difference between the two curves.
- `return total / n_inputs` — the average magnitude at each depth.
- `plt.yscale("log")` — a log axis, so a 30-fold drop is visible next to a 10% drift.
- `plt.title(...)` — the live readout of both final magnitudes.
- `interact(show, depth=(1, 24, 1))` — the slider, its range and step.

## 5. When it breaks: no residual, no signal

Delete the two additions and each block replaces the stream with its own small output. The length of the activation falls by more than an order of magnitude, and every further block simply re-normalises that shrunken vector.

In [ ]:
depth = 24                                            # how deep we go
w = trace(depth, True)                                # curve with the shortcut
o = trace(depth, False)                               # curve without it
depths = np.arange(depth + 1)                         # 0, 1, 2, ... 24

for k in [0, 1, 2, 6, 12, 24]:                        # a few depths to inspect
    print(f"depth {k:2d}: with {w[k]:6.2f}   without {o[k]:6.3f}   ratio {o[k] / w[k]:.3f}")   # the table

plt.figure(figsize=(6, 3))                            # a wide, short figure
plt.plot(depths, w, marker="o", label="with residual")     # stays near 2.9
plt.plot(depths, o, marker="o", label="without residual")  # drops and stays down
plt.yscale("log")                                     # the gap spans three factors
plt.xlabel("block")                                   # x-axis label
plt.ylabel("magnitude (log scale)")                   # y-axis label
plt.legend()                                          # name the two lines
plt.title("without the shortcut, 99% of the signal is gone in one block")   # the takeaway
plt.show()                                            # render it

print("without residual, |stream| is", round(o[-1] / w[0], 4), "of the input")   # the damage

hh = np.random.default_rng(100).normal(size=(2, d))   # one input, the seed used in section 4
for _ in range(depth):                                # push it through unchanged blocks
    a = attn(ln(hh))                                  # attention branch
    hh = a                                            # no residual: the branch replaces the stream
    f = ffn(ln(hh))                                   # feed-forward branch
    hh = f                                            # replaced again
print("token norms after 24 blocks:", np.round(np.linalg.norm(hh, axis=1), 4))   # both tiny

**Line by line**

- `depth = 24` — deep enough for the collapse to flatten out.
- `w`, `o` — the averaged curves from the helper defined in section 4.
- `depths = np.arange(depth + 1)` — one x value per depth, starting at 0.
- `for k in [0, 1, 2, 6, 12, 24]` — print a small table at chosen depths.
- `print(f"...")` — the ratio column makes the collapse quantitative.
- `plt.plot(depths, o, ...)` — the no-residual curve on the same axes.
- `plt.yscale("log")` — log scale, because the drop is multiplicative.
- `plt.xlabel` / `plt.ylabel` — name the two axes.
- `plt.title(...)` — the takeaway, in one sentence.
- `o[-1] / w[0]` — the surviving fraction of the original input magnitude.
- `for _ in range(depth)` — replay one input through the no-residual block.
- `hh = a` then `hh = f` — each branch overwrites the stream instead of adding to it.
- `np.linalg.norm(hh, axis=1)` — one length per token, printed at the end.

## Your turn

Set `Wo` and `W2` to `scale=0.5` and rerun sections 3 and 5: the branch now matches the stream, so both curves change shape. Predict first whether the no-residual line still collapses, then explain the answer in terms of what `h = h + a` preserves that `h = a` does not.